# Burn probability data

Using [Open Climate Risk data](https://source.coop/carbonplan/carbonplan-ocr). Retrieval adapted from [their documentation](https://docs.carbonplan.org/ocr/en/latest/how-to/work-with-data.html#raster-xarray).


In [1]:
import icechunk
import xarray as xr

# Configure S3 storage for the Icechunk repository
version = "v1.1.0"
storage = icechunk.s3_storage(
    bucket="us-west-2.opendata.source.coop",
    prefix=f"carbonplan/carbonplan-ocr/output/fire-risk/tensor/production/{version}/ocr.icechunk",
    region="us-west-2",
    anonymous=True,
)

# Open the repository
repo = icechunk.Repository.open(storage)

# Create a read-only session on the main branch
session = repo.readonly_session("main")

## Open the dataset


In [ ]:
ds = xr.open_dataset(session.store, engine="zarr", chunks={})
ds

/tmp/ipykernel_2394/2172365150.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "latitude" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(session.store, engine="zarr", chunks=10000)  # chunks={}
/tmp/ipykernel_2394/2172365150.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "longitude" starting at index 10000. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(session.store, engine="zarr", chunks=10000)  # chunks={}


<xarray.Dataset> Size: 652GB
Dimensions:        (latitude: 97579, longitude: 208881)
Coordinates:
  * latitude       (latitude) float64 781kB 22.43 22.43 22.43 ... 52.48 52.48
  * longitude      (longitude) float64 2MB -128.4 -128.4 ... -64.05 -64.05
Data variables:
    bp_2011        (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    bp_2047        (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    bp_2011_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    crps_scott     (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    rps_2011       (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    rps_scott      (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    bp_2047_riley  (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
    rps_2047       (latitude, longitude) float32 82GB dask.array<chunksize=(10000, 10000), meta=np.ndarray>
Attributes:
    version:          1.1.0
    provider:         CarbonPlan
    terms_of_access:  https://docs.carbonplan.org/ocr/en/latest/terms-of-data...
    data_sources:     https://docs.carbonplan.org/ocr/en/latest/reference/dat...
    license_name:     CC-BY-4.0
    license_url:      https://creativecommons.org/licenses/by/4.0/

Getting the ["Annual burn probability for ~2011 climate conditions"](https://docs.carbonplan.org/ocr/en/latest/reference/data-schema.html#core-risk-variables):


In [ ]:
burn_prob = ds["bp_2011"]
burn_prob

<xarray.DataArray 'bp_2011' (latitude: 32468, longitude: 35715)> Size: 5GB
dask.array<getitem, shape=(32468, 35715), dtype=float32, chunksize=(10000, 10000), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 260kB 32.0 32.0 32.0 32.0 ... 42.0 42.0 42.0
  * longitude  (longitude) float64 286kB -125.0 -125.0 -125.0 ... -114.0 -114.0

These are ["gridded geospatial layers stored at 30m resolution"](https://docs.carbonplan.org/ocr/en/latest/reference/data-schema.html#raster-tensor-datasets). Downsample to 1km:


In [5]:
# Approximate 1 km aggregation from the native ~30 m grid
factor = 33
square_size_m = 30 * factor

burn_prob_1km = burn_prob.coarsen(
    latitude=factor,
    longitude=factor,
    boundary="trim",
).mean()

burn_prob_1km

<xarray.DataArray 'bp_2011' (latitude: 983, longitude: 1082)> Size: 4MB
dask.array<mean_agg-aggregate, shape=(983, 1082), dtype=float32, chunksize=(303, 303), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 8kB 32.01 32.02 32.03 ... 41.97 41.98 41.99
  * longitude  (longitude) float64 9kB -125.0 -125.0 -125.0 ... -114.0 -114.0

Store in Parquet for reading by DuckDB:


In [6]:
from pathlib import Path

import dask.dataframe

PARQUET_DIR = Path("data/burn_prob_1km")

ddf: dask.dataframe.DataFrame = burn_prob_1km.to_dask_dataframe()
ddf.to_parquet(PARQUET_DIR, write_index=False)

Number of rows:


In [16]:
ddf.shape[0].compute()

1063606